# index_momentum 因子：只用指数自身路径

仿 `session_lab.ipynb`：选一个宽基指数、一天，加载 warmup 后画出当日 K 线。
图的结构与 HTML 报告相同（K 线、成交量、MACD），并在 **MACD 正下方** 多一行，叠加
六个已经在随机游走 σ 单位里的因子。最底下一行是它们的等权平均 `score`。

**没有横截面。** 不看其他个股，也不对宇宙做 z-score。单标的时 score 仍然有定义。

逻辑都在 `src/qtrader.strategies.index_momentum`：工作台是 `SessionLab`，图是
`price_chart(..., extra_panel=...)`。改 `SYMBOL` / `DAY` 后重新跑加载单元格即可。

`SYMBOL` 可以是 `QQQ`、`SPY`，或宇宙外的 `Nikkei`（解析为 `EWJ`）。
**Warm-up 会自动加载。** `rvol` 的分母要 10 个完整历史交易日。只报告、只画你选的那一天。


In [1]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "src" / "qtrader").is_dir() and (_p / "config").is_dir():
        REPO_ROOT = _p
        break
else:
    raise ModuleNotFoundError(
        "Cannot find the qtrader repo. Start Jupyter from the repo, or run "
        "`pip install -e .` in /Users/zihao/work/quant_dev"
    )

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
import plotly.io as pio

from qtrader.experiments.session_lab import SessionLab
from qtrader.strategies.index_momentum import FACTORS
from qtrader.viz.charts import price_chart

pio.renderers.default = "notebook"
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.3f}".format)
print(f"cwd = {Path.cwd()}")


cwd = /Users/zihao/work/quant_dev


## 1. 选一天

`CONFIG` 决定宇宙、成本和因子窗口。默认是 1 分钟网格上的 `index_momentum`，宇宙是 `SPY` + `QQQ`。
构建 lab 会读 parquet；本地没有的标的会向 Alpaca 下载，名字不存在才报错。
`SYMBOL` 可以是宇宙外的名字，例如 `Nikkei`（解析为 `EWJ`）。


In [17]:
CONFIG = "config/backtest/index_momentum_1min.yaml"
SYMBOL = "QQQ"
DAY    = "2026-08-11"

lab = SessionLab(CONFIG, SYMBOL, DAY)
print(
    f"{len(lab.session_index)} bars on {DAY}, "
    f"{lab.window[0].tz_convert('America/New_York'):%H:%M}"
    f" to {lab.window[1].tz_convert('America/New_York'):%H:%M} ET"
)


390 bars on 2026-08-11, 09:30 to 15:59 ET


## 2. 因子定义

六个有方向的量都已经除过 `σ`（分时季节性波动，用该标的自己的历史），单位相同，直接等权平均成 `score`。
丢掉了 1 分钟收益（IEX 噪声）、30 分钟收益（和后半天的 `session_z` 重复）、以及任何相对其他名字的量。

| 因子 | 定义 | 进入 score |
| --- | --- | --- |
| `session_z` | 开盘至今漂移 / `(σ√n)`。窗口随交易日变长 | 等权 |
| `ewma_z` | session 内重启的 EWMA 漂移 z。开盘大波动后可以掉头 | 等权 |
| `ret_5_z` / `ret_15_z` | 过去 5 / 15 根 session 内 log return / `(σ√n)` | 等权 |
| `vwap_z` | `(close / 当日 VWAP − 1) / (σ√n)`，与 `session_z` 同一单位 | 等权 |
| `macd_z` | session 内重启的 MACD 直方图速度，随机游走 σ | 等权 |
| `rvol` | 近 5 根成交量 / 同分钟历史均值 | **不进加权和**。无方向，只作 `min_rvol` 门 |
| `er_15` | 带符号效率比 | **不进加权和**。量纲 [-1, 1]，默认也不做门 |

每个因子在 bar `t` 只用该标的 `<= t` 的数据。决策在该 bar 收盘，成交在下一根开盘。
个股横截面动量在 R04/R05 上是反转的；这是指数自身路径的另一项主张，符号不要预先取负。

### 开仓（config 起始值，不是搜索结果）

多空对称。决策在 bar `t` 收盘，成交在 `t+1` 开盘。**同一根不反手**：先平仓，下一根才能反向开。

做多（做空把不等号反过来）需要同时满足：

1. 当前空仓
2. 时钟：`09:40 ≤ t < 15:30`（时间戳是 bar **开**盘；09:40 决策 → 最早 09:41 成交），且未到 `15:50` 强平
3. 该分钟可交易、有成交、价格有限、`score` 有限（至少 4 个因子已算出）
4. `rvol ≥ 1.0`（该分钟成交量不少于同分钟历史均值）。`min_er` 默认关闭
5. `score > 1.5`（空头 `score < −1.5`）
6. 账户未满 `max_positions=2`；同一根多个候选按 `|score|` 从大到小占坑
7. 距离上一次平仓已过 `reentry_cooldown_bars=15` 根

平仓：`score` 穿越 0（且已持有至少 10 根；ATR 止损不受这个限制），或触及 `2×ATR` 止损，或 `15:50` 强平，或不隔夜。
**没有时间上限。** 以前的 30 根硬切会把同一段趋势拆成十几笔。


## 3. 跑策略，取出当日因子

`lab.run()` 在已缓存的 panel 上重算信号，不重新读盘。阈值用 config 默认值即可：
这里要看的是因子轨迹，不是交易结果。`gross_bps` 在 0 笔交易时仍是 NaN，那是空交易簿的汇总。


In [18]:
run = lab.run()
print(lab.summary(run).round(4).to_string())
print(
    "(gross_bps / hit_rate 为 NaN 表示这一天这个标的 0 笔交易，不是因子算错。)"
)

eligible = lab.context.tradable.loc[list(lab.session_index)]
print(
    f"symbol={lab.symbol}  eligible {int(eligible[lab.symbol].mean()*100)}% of bars"
)

indicators = run.result.signals.indicators[lab.symbol].loc[list(lab.session_index)]
missing = [c for c in FACTORS if c not in indicators.columns]
if missing:
    raise KeyError(f"strategy did not publish {missing}")
print("factors:", list(FACTORS))
print(indicators[list(FACTORS) + ["score", "rvol", "er_15"]].describe().T[["count", "mean", "std", "min", "max"]])

trades = lab.trades(run)
if trades.empty:
    print("no trades")
else:
    hold_min = (trades["exit_time"] - trades["entry_time"]).dt.total_seconds() / 60
    print("hold minutes:", hold_min.round(0).astype(int).tolist())


trades        5.000
net_pnl      -6.842
gross_bps     1.215
cost_bps      1.500
hit_rate      0.400
day_return   -0.001
(gross_bps / hit_rate 为 NaN 表示这一天这个标的 0 笔交易，不是因子算错。)
symbol=QQQ  eligible 100% of bars
factors: ['session_z', 'ewma_z', 'ret_5_z', 'ret_15_z', 'vwap_z', 'macd_z']
            count   mean   std    min   max
session_z 389.000 -1.659 1.231 -6.553 0.221
ewma_z    389.000 -0.239 1.419 -7.359 5.848
ret_5_z   386.000 -0.162 1.465 -5.899 4.891
ret_15_z  376.000 -0.215 1.373 -5.776 4.230
vwap_z    390.000 -0.768 0.850 -8.778 0.672
macd_z    389.000 -0.062 1.389 -4.650 5.113
score     389.000 -0.517 0.930 -5.279 2.141
rvol      390.000  0.944 0.482  0.199 2.617
er_15     375.000 -0.044 0.319 -0.751 0.742
hold minutes: [10, 2, 23, 3, 40]


## 4. 当日图：K 线 / 成交量 / MACD / 因子 / score / 收益率

`window=lab.window` 切掉 warmup，只留这一天。MACD 下面那行是六条 own-path 因子，零轴是「相对随机游走为零」。
一条因子到 ±2，表示这个指数自身刚走出一段约两倍 σ 的漂移，不是相对其他股票的排名。
最底下一行是**当日收益率**：虚线是这个指数从开盘买到现在，实线是策略（已扣成本），都从这一天的第一根重定基为 0。


In [19]:
fig = price_chart(
    run.result, lab.symbol,
    window=lab.window,
    height=1280,
    extra_panel=list(FACTORS),
    extra_panel_title="Factors (own-path σ)",
    return_panel=True,
)
fig.show()


## 5. 开盘后前 30 根

把图上开盘附近的因子读成表。时间是 America/New_York。
`macd_z` 在 EMA 未热时会是 NaN；`rvol` 在不满 10 个历史交易日时会是 NaN。早盘 score 由 `session_z` / `ewma_z` / `vwap_z` 撑着。


In [5]:
OPEN_BARS = 30

session = indicators.copy()
session.index = session.index.tz_convert("America/New_York")
early = session.iloc[:OPEN_BARS][list(FACTORS) + ["score", "rvol"]]
early.index = early.index.strftime("%H:%M")
early.round(2)


,session_z,ewma_z,ret_5_z,ret_15_z,vwap_z,macd_z,score,rvol
timestamp,,,,,,,,
09:30,NaN,NaN,NaN,NaN,-6.140,NaN,NaN,1.040
09:31,1.200,1.200,NaN,NaN,0.460,1.200,1.010,2.880
09:32,1.440,1.370,NaN,NaN,0.670,1.170,1.160,3.340
09:33,1.680,1.410,NaN,NaN,0.590,0.710,1.100,2.080
09:34,1.660,1.410,1.480,NaN,0.760,0.700,1.200,1.630
09:35,1.070,0.850,1.070,NaN,0.490,0.200,0.740,1.360
09:36,0.830,0.430,0.150,NaN,0.250,-0.590,0.210,0.760
09:37,1.470,1.100,0.410,NaN,0.630,0.120,0.750,0.790
09:38,0.850,0.440,0.180,NaN,0.280,-0.490,0.250,0.740
